In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm
import os
import gc

In [ ]:
# 1. Path Configuration - Local Cluster Setup
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.abspath(os.path.join(SCRIPT_DIR, ".."))

INPUT_FILE = os.path.join(PROJECT_ROOT, "data", "baseline_results.csv")
OUTPUT_FILE = os.path.join(PROJECT_ROOT, "data", "verified_baseline.csv")

In [ ]:
def run_verification(model_id="meta-llama/Meta-Llama-3.1-70B-Instruct"):
    print(f"--- Loading STRONG JUDGE: {model_id} ---")
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto", 
        trust_remote_code=True
    )

    if not os.path.exists(INPUT_FILE):
        print(f"ERROR: Could not find {INPUT_FILE}!")
        return

    df = pd.read_csv(INPUT_FILE)
    verified_results = []

    print(f"--- Starting Verification Process for {len(df)} samples ---")
    
    for index, row in tqdm(df.iterrows(), total=len(df)):
        ground_truth_block = f"""
        - Vulnerability Type: {row.get('cwe', 'Unknown')}
        - Reference ID: {row.get('cve', 'Unknown')}
        - Developer Commit Message: {row.get('truth_description', 'No description available')}
        """

        judge_prompt = f"""[INST] You are a Senior Security Auditor. 
            Compare the AI EXPLANATION against the GROUND TRUTH.

            CODE: 
            {row['code']}

            GROUND TRUTH: 
            {ground_truth_block}

            AI EXPLANATION: 
            {row['baseline_explanation']}

            Does the AI EXPLANATION identify the same security vulnerability and root cause as the GROUND TRUTH?
            Answer ONLY 'YES' or 'NO' followed by a short reason. [/INST]"""
        
        inputs = tokenizer(judge_prompt, return_tensors="pt", truncation=True, max_length=3500).to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=150, 
                temperature=0.1, 
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        judge_response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        is_correct = judge_response.strip().upper().startswith("YES")

        verified_results.append({
            **row.to_dict(),
            'judge_response': judge_response,
            'is_correct': is_correct
        })

        del inputs
        del outputs
        torch.cuda.empty_cache()
        gc.collect()

    # Save Results
    output_df = pd.DataFrame(verified_results)
    output_df.to_csv(OUTPUT_FILE, index=False)
    
    print(f"\n--- Verification Complete! {output_df['is_correct'].sum()}/{len(df)} passed. ---")

In [ ]:
if __name__ == "__main__":
    run_verification()